# NMF on ModulAir 00690



In [1]:
from sklearn.decomposition import NMF
import numpy as np
import pandas as pd

## Cleaning Data

In [2]:
#importing data from Modulair MOD-00690
data = pd.read_csv('MOD-00690.csv').set_index('timestamp')
data.head()

,id,timestamp_local,sn,rh,temp,bin0,bin1,bin2,bin3,bin4,...,no2,o3,pm1_model_id,pm25_model_id,pm10_model_id,co_model_id,no_model_id,no2_model_id,o3_model_id,ws_scalar
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-12-31T23:59:44Z,577611177,2025-12-31T18:59:44Z,MOD-00690,49.1,0.3,5.980,0.866,0.322,0.064,0.054,...,30.015,32.285,14343,14344,14345,14477.0,14502.0,14552.0,14527.0,3.15
2025-12-31T23:58:44Z,577611176,2025-12-31T18:58:44Z,MOD-00690,49.5,0.3,6.039,0.852,0.284,0.055,0.046,...,30.472,32.252,14343,14344,14345,14477.0,14502.0,14552.0,14527.0,3.04
2025-12-31T23:57:44Z,577611175,2025-12-31T18:57:44Z,MOD-00690,49.6,0.3,5.887,0.878,0.307,0.038,0.049,...,30.942,31.893,14343,14344,14345,14477.0,14502.0,14552.0,14527.0,3.04
2025-12-31T23:56:44Z,577611174,2025-12-31T18:56:44Z,MOD-00690,49.5,0.3,6.183,0.918,0.321,0.041,0.026,...,30.946,32.603,14343,14344,14345,14477.0,14502.0,14552.0,14527.0,1.16
2025-12-31T23:55:44Z,577609139,2025-12-31T18:55:44Z,MOD-00690,49.4,0.3,6.126,0.839,0.291,0.045,0.033,...,30.713,32.260,14343,14344,14345,14477.0,14502.0,14552.0,14527.0,1.50


In [3]:
#only columns of interest
COLS_TO_INCLUDE = ['timestamp_local','co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
data = data[COLS_TO_INCLUDE]

data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
timestamp,,,,,,,,,,,
2025-12-31T23:59:44Z,2025-12-31T18:59:44Z,754.985,2.521,30.015,32.285,5.980,0.866,0.322,0.064,0.054,0.027
2025-12-31T23:58:44Z,2025-12-31T18:58:44Z,756.754,2.768,30.472,32.252,6.039,0.852,0.284,0.055,0.046,0.006
2025-12-31T23:57:44Z,2025-12-31T18:57:44Z,771.872,2.624,30.942,31.893,5.887,0.878,0.307,0.038,0.049,0.006
2025-12-31T23:56:44Z,2025-12-31T18:56:44Z,787.131,2.624,30.946,32.603,6.183,0.918,0.321,0.041,0.026,0.019
2025-12-31T23:55:44Z,2025-12-31T18:55:44Z,776.529,2.520,30.713,32.260,6.126,0.839,0.291,0.045,0.033,0.029


In [4]:
#removing the UTC time
data = data.reset_index(drop = True)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-31T18:59:44Z,754.985,2.521,30.015,32.285,5.980,0.866,0.322,0.064,0.054,0.027
1,2025-12-31T18:58:44Z,756.754,2.768,30.472,32.252,6.039,0.852,0.284,0.055,0.046,0.006
2,2025-12-31T18:57:44Z,771.872,2.624,30.942,31.893,5.887,0.878,0.307,0.038,0.049,0.006
3,2025-12-31T18:56:44Z,787.131,2.624,30.946,32.603,6.183,0.918,0.321,0.041,0.026,0.019
4,2025-12-31T18:55:44Z,776.529,2.520,30.713,32.260,6.126,0.839,0.291,0.045,0.033,0.029


In [5]:
#converting to datetime
data['timestamp_local'] = pd.to_datetime(data['timestamp_local'],
                                       format='%Y-%m-%dT%H:%M:%SZ',
                                       exact=False)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-31 18:59:44,754.985,2.521,30.015,32.285,5.980,0.866,0.322,0.064,0.054,0.027
1,2025-12-31 18:58:44,756.754,2.768,30.472,32.252,6.039,0.852,0.284,0.055,0.046,0.006
2,2025-12-31 18:57:44,771.872,2.624,30.942,31.893,5.887,0.878,0.307,0.038,0.049,0.006
3,2025-12-31 18:56:44,787.131,2.624,30.946,32.603,6.183,0.918,0.321,0.041,0.026,0.019
4,2025-12-31 18:55:44,776.529,2.520,30.713,32.260,6.126,0.839,0.291,0.045,0.033,0.029


In [6]:
#taking hourly average of df. round to floor of the hour
data = data.groupby(data['timestamp_local'].dt.floor('h')).agg(co = ('co','mean'),
                                                         no2 = ('no2','mean'),
                                                         o3 = ('o3','mean'),
                                                         no = ('no','mean'),
                                                         bin0 = ('bin0','mean'),
                                                         bin1 = ('bin1','mean'),
                                                         bin2 = ('bin2','mean'),
                                                         bin3 = ('bin3','mean'),
                                                         bin4 = ('bin4','mean'),
                                                         bin5 = ('bin5','mean')).reset_index()

data = data.round(decimals = 2)
data = data.dropna()
data

,timestamp_local,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-03-31 20:00:00,855.71,31.10,37.51,1.98,19.70,2.58,1.22,0.46,0.64,0.52
1,2025-03-31 21:00:00,961.80,32.43,37.81,2.82,27.75,5.16,1.78,0.56,0.74,0.64
2,2025-03-31 22:00:00,921.55,22.91,44.66,2.86,23.44,5.36,1.69,0.46,0.54,0.41
3,2025-03-31 23:00:00,779.51,16.17,54.80,2.84,14.97,3.79,1.11,0.27,0.29,0.19
4,2025-04-01 00:00:00,708.09,14.86,59.63,2.78,14.08,3.26,0.90,0.21,0.23,0.14
...,...,...,...,...,...,...,...,...,...,...,...
6516,2025-12-31 14:00:00,750.04,28.94,36.36,2.01,5.46,0.54,0.16,0.03,0.02,0.01
6517,2025-12-31 15:00:00,774.67,29.42,36.59,1.94,4.88,0.55,0.16,0.03,0.02,0.01
6518,2025-12-31 16:00:00,786.70,30.03,34.84,2.50,5.25,0.64,0.18,0.03,0.03,0.01
6519,2025-12-31 17:00:00,786.94,30.59,32.99,2.48,6.33,0.84,0.25,0.05,0.04,0.02


In [7]:
#setting local time as index
data = data.set_index('timestamp_local')
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-03-31 20:00:00,855.71,31.10,37.51,1.98,19.70,2.58,1.22,0.46,0.64,0.52
2025-03-31 21:00:00,961.80,32.43,37.81,2.82,27.75,5.16,1.78,0.56,0.74,0.64
2025-03-31 22:00:00,921.55,22.91,44.66,2.86,23.44,5.36,1.69,0.46,0.54,0.41
2025-03-31 23:00:00,779.51,16.17,54.80,2.84,14.97,3.79,1.11,0.27,0.29,0.19
2025-04-01 00:00:00,708.09,14.86,59.63,2.78,14.08,3.26,0.90,0.21,0.23,0.14


In [8]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()

In [9]:
#scaling
def maximum_absolute_scaling(df):
    # copy the dataframe
    df_scaled = df.copy()
    # apply maximum absolute scaling
    for column in df_scaled.columns:
        df_scaled[column] = df_scaled[column]  / df_scaled[column].abs().max()
    return df_scaled

# call the maximum_absolute_scaling function
data = maximum_absolute_scaling(data)

In [10]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-03-31 20:00:00,0.554752,0.601199,0.393187,0.024664,0.266144,0.095946,0.095090,0.133333,0.246154,0.335484
2025-03-31 21:00:00,0.623529,0.626909,0.396331,0.035127,0.374899,0.191893,0.138737,0.162319,0.284615,0.412903
2025-03-31 22:00:00,0.597435,0.442876,0.468134,0.035625,0.316671,0.199331,0.131723,0.133333,0.207692,0.264516
2025-03-31 23:00:00,0.505352,0.312585,0.574423,0.035376,0.202243,0.140945,0.086516,0.078261,0.111538,0.122581
2025-04-01 00:00:00,0.459051,0.287261,0.625052,0.034629,0.190219,0.121235,0.070148,0.060870,0.088462,0.090323


In [11]:
data.to_csv(r'MOD-00690_timeseries_hourly_scaled.csv')

In [12]:
# Check start date of df to make sure that it's loaded in correctly
start = data.index.min()

end = data.index.min()

print(start, end)

2025-03-31 20:00:00 2025-03-31 20:00:00


## Implementing NMF

In [13]:
#setting up the NMF
nmf = NMF(n_components = 4, max_iter = 8000)

In [14]:
W = nmf.fit_transform(data)
H = nmf.components_
V = pd.DataFrame(np.dot(W,H), columns=data.columns)
V.index = data.index
V

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-03-31 20:00:00,0.561582,0.597778,0.399195,0.032176,0.201214,0.199786,0.163878,0.164918,0.221876,0.238184
2025-03-31 21:00:00,0.627464,0.624761,0.404665,0.034787,0.314582,0.281458,0.216521,0.210529,0.275720,0.292512
2025-03-31 22:00:00,0.584389,0.447820,0.479408,0.033752,0.300523,0.229320,0.165493,0.155482,0.203800,0.217930
2025-03-31 23:00:00,0.523433,0.305559,0.564062,0.032508,0.200865,0.127057,0.087847,0.080973,0.112157,0.124856
2025-04-01 00:00:00,0.512602,0.266292,0.594579,0.032533,0.178526,0.100753,0.066994,0.060511,0.086811,0.099110
...,...,...,...,...,...,...,...,...,...,...
2025-12-31 14:00:00,0.469529,0.565970,0.389745,0.028927,0.079902,0.023604,0.008687,0.004641,0.010360,0.015963
2025-12-31 15:00:00,0.476967,0.578583,0.396688,0.029423,0.076583,0.021876,0.008051,0.004411,0.010488,0.016316
2025-12-31 16:00:00,0.477997,0.593179,0.382534,0.029274,0.084163,0.025846,0.009512,0.004984,0.010177,0.015478


In [15]:
W_df = pd.DataFrame(W, columns =[f'Feature {i+1}' for i in range(0,4)]) #array-like of shape (n_samples, n_components)
W_df

,Feature 1,Feature 2,Feature 3,Feature 4
0,0.036225,0.108895,0.049708,0.107585
1,0.032180,0.114561,0.082250,0.134477
2,0.042934,0.079396,0.076313,0.096565
3,0.058074,0.050257,0.045558,0.048922
4,0.062910,0.042124,0.038397,0.035620
...,...,...,...,...
6414,0.040135,0.102427,0.014963,0.000000
6415,0.041023,0.104710,0.013868,0.000000
6416,0.038915,0.107782,0.016385,0.000000
6417,0.035997,0.110260,0.020716,0.002718


In [16]:
COLS_TO_INCLUDE.pop(0)
COLS_TO_INCLUDE

['co', 'no', 'no2', 'o3', 'bin0', 'bin1', 'bin2', 'bin3', 'bin4', 'bin5']

In [17]:
H_df = pd.DataFrame(H, index = [f'Feature {i+1}' for i in range(0,4)], columns = COLS_TO_INCLUDE) #array-like of shape (n_components, n_features)
H_df

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Feature 1,5.248330,0.717633,8.539338,0.371048,0.661336,0.000000,0.000000,0.014799,0.231822,0.397734
Feature 2,2.221500,5.244408,0.286709,0.122072,0.000000,0.000000,0.000000,0.005402,0.000000,0.000000
Feature 3,2.094796,0.000000,1.179581,0.102375,3.565959,1.577418,0.580565,0.233516,0.070554,0.000000
Feature 4,0.236302,0.006435,0.000000,0.003283,0.000000,1.128186,1.255000,1.414567,1.951679,2.079999


In [18]:
#converting the results to a dataframe
results = pd.DataFrame(W,index=data.index) #H is time series data of the factors, W is the comp (coeff matrix)
results.columns = ["Factor {}".format(i+1) for i in range(H.T.shape[1])]
results

,Factor 1,Factor 2,Factor 3,Factor 4
timestamp_local,,,,
2025-03-31 20:00:00,0.036225,0.108895,0.049708,0.107585
2025-03-31 21:00:00,0.032180,0.114561,0.082250,0.134477
2025-03-31 22:00:00,0.042934,0.079396,0.076313,0.096565
2025-03-31 23:00:00,0.058074,0.050257,0.045558,0.048922
2025-04-01 00:00:00,0.062910,0.042124,0.038397,0.035620
...,...,...,...,...
2025-12-31 14:00:00,0.040135,0.102427,0.014963,0.000000
2025-12-31 15:00:00,0.041023,0.104710,0.013868,0.000000
2025-12-31 16:00:00,0.038915,0.107782,0.016385,0.000000


In [19]:
COLS_TO_INCLUDE = ['co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
comp = pd.DataFrame(H, index = results.columns, columns = COLS_TO_INCLUDE)
comp

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Factor 1,5.248330,0.717633,8.539338,0.371048,0.661336,0.000000,0.000000,0.014799,0.231822,0.397734
Factor 2,2.221500,5.244408,0.286709,0.122072,0.000000,0.000000,0.000000,0.005402,0.000000,0.000000
Factor 3,2.094796,0.000000,1.179581,0.102375,3.565959,1.577418,0.580565,0.233516,0.070554,0.000000
Factor 4,0.236302,0.006435,0.000000,0.003283,0.000000,1.128186,1.255000,1.414567,1.951679,2.079999


In [20]:
res = []

for col in comp.columns:
    #for each factor, compute its contribution to the species in V
    by_factor = pd.Series(dtype=float)
    for i, factor in enumerate(results.columns):
        contrib = H[i, data.columns.get_loc(col)] * W[:, i].sum()
        by_factor.at[factor] = contrib

    #normalizing by the total amount of that species in the original data
    by_factor /= data[col].sum()

    #adding as a row to the resulting dataframe
    res.append(pd.DataFrame(by_factor).T.rename(index={0: col}))

res = pd.concat(res)
res.columns = results.columns
res['Residual'] = 1 - res.sum(axis=1)

res

,Factor 1,Factor 2,Factor 3,Factor 4,Residual
co,0.533995,0.330102,0.119470,0.006776,0.009657
no,0.606135,0.291234,0.093742,0.001512,0.007377
no2,0.086036,0.918248,0.000000,0.000217,-0.004502
o3,0.892617,0.043769,0.069114,0.000000,-0.005500
bin0,0.252886,0.000000,0.764329,0.000000,-0.017215
bin1,0.000000,0.000000,0.806514,0.290045,-0.096558
bin2,0.000000,0.000000,0.505159,0.549086,-0.054245
bin3,0.026775,0.014274,0.236813,0.721326,0.000814
bin4,0.284084,0.000000,0.048463,0.674092,-0.006639
bin5,0.410011,0.000000,0.000000,0.604345,-0.014356


In [22]:
results.to_csv(r'MOD-00690_4_factor_results.csv')
comp.to_csv(r'MOD-00690_4_factor_comp.csv')
res.to_csv(r'MOD-00690_4_factor_resid.csv')

In [23]:
# check how many rows of data you have
len(data)

6419